# Day 2, Notebook 2: cross the file boundary

Restart the kernel and your records are gone. That is not a bug in your code. It is what memory means.

This notebook takes the functions you carved in notebook 1 and points them at real files, in both directions.

Position in the day:

`[inline cell] > [packaged decision] > [read the failure] > [log the rejection] > **[cross the boundary]**`

The map below is where this notebook sits in the day and what it adds. Every
notebook in the programme opens on the same pair, so you know where you are before you read a line.

The cell that draws it also brings in the programme's helper. `scripts/c2kit.py` is found by
walking up from this notebook's own folder, which is what lets the same file run whether you
pressed Run All here or a script ran it for you. The helper loads the day's data from `../data/`,
draws every diagram you see in these notebooks, and runs the checks that tell you a cell did what
it claimed.

In [1]:
import sys, pathlib

here = pathlib.Path.cwd()
for parent in [here, *here.parents]:
    if (parent / "scripts" / "c2kit.py").exists():
        sys.path.insert(0, str(parent / "scripts"))
        break
import c2kit as kit

kit.side_by_side(
    kit.ladder(["functions and errors", "files and formats", "hands-on: trace the calls", "hands-on: the truncated feed"], lit=1, title="the day's notebooks", show=False),
    kit.flow(["the wrong path", "read a CSV by name", "convert or reject", "one pass, two files out", "a different agreement", "when JSON goes wrong"], title="what this notebook adds", show=False),
)

## Setup

Everything this notebook needs, in one cell at the top, so it runs cold in a fresh Codespace.

In [2]:
import csv
import json
import os
import traceback

# csv and json are today's topic. os and traceback are plumbing: one makes a folder,
# the other lets a failure print itself without stopping the notebook. Not topics today.

DATA_DIR = "../data"
ORDERS_CSV = f"{DATA_DIR}/C2_W01_D02_orders_STUDENT.csv"
ORDERS_JSON = f"{DATA_DIR}/C2_W01_D02_orders_STUDENT.json"
TRUNCATED_JSON = f"{DATA_DIR}/C2_W01_D02_vendor_truncated_STUDENT.json"
OUTPUT_DIR = "output"

os.makedirs(OUTPUT_DIR, exist_ok=True)

def show_failure(fn):
    """Run something that is meant to fail and print its real traceback."""
    try:
        fn()
    except Exception:
        print(traceback.format_exc())

print("setup done, writing outputs into", OUTPUT_DIR)

setup done, writing outputs into output


In [3]:
kit.flow(["the wrong path", "read a CSV by name", "convert or reject", "one pass, two files out", "a different agreement", "when JSON goes wrong"], lit=0)

## Section 1: the wrong path, on purpose

Before opening a file correctly, look at what a wrong path says. This is the cheapest error in the whole programme.

In [4]:
def open_a_wrong_path():
    return open("data/orderz.csv")

show_failure(open_a_wrong_path)

Traceback (most recent call last):
  File "/tmp/ipykernel_5185/1347761746.py", line 20, in show_failure
    fn()
  File "/tmp/ipykernel_5185/3874348740.py", line 2, in open_a_wrong_path
    return open("data/orderz.csv")
           ^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/IPython/core/interactiveshell.py", line 339, in _modified_open
    return io_open(file, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: 'data/orderz.csv'



```
FileNotFoundError: [Errno 2] No such file or directory: 'data/orderz.csv'
```

It names the exact path it tried. Read that path out loud and you find the typo yourself, without a search engine.

Two things worth knowing before you edit anything:

1. The path is relative to where the kernel is running, not to where the file browser is pointing.
2. `Errno 2` means the file was not there. A different errno means a different problem, for example a permission you do not have.

In [5]:
print("the kernel is running in:", os.getcwd())
print("so 'data/orderz.csv' meant that folder, plus data, plus orderz.csv")

the kernel is running in: /home/user/c2-content-factory/content/W01/D2/notebooks
so 'data/orderz.csv' meant that folder, plus data, plus orderz.csv


In [6]:
import pathlib as _pl
kit.check("this notebook runs with its own folder as the working directory",
          _pl.Path.cwd().name == "notebooks", f"cwd is {_pl.Path.cwd().name}")
kit.check("so the day's data sits one level up, in ../data",
          (_pl.Path.cwd().parent / "data").is_dir())

### Where a relative path is counted from

In [7]:
kit.vflow(["the notebook you opened",
           "its own folder, notebooks/",
           "..  goes up one, to the day folder",
           "../data/ is where the day's files live",
           "open('data/orderz.csv') looked inside notebooks/ and found nothing"],
          lit=4, title="why the wrong path said what it said")

In [8]:
kit.flow(["the wrong path", "read a CSV by name", "convert or reject", "one pass, two files out", "a different agreement", "when JSON goes wrong"], lit=1)

## Section 2: reading a CSV by name

Section 1 plus one new element: the file actually opens.

A file format is an agreement about structure. CSV agrees about three things and no more: one row per record, commas between fields, and the first row names the fields. Nothing in that agreement mentions types.

```
the file            what Python receives
4500          ->    "4500"
twelve        ->    "twelve"
(empty cell)  ->    ""
```

In [9]:
with open(ORDERS_CSV) as f:
    orders = list(csv.DictReader(f))

print(f"{len(orders)} records")
print(orders[0])
print()
print("every value is a", type(orders[0]["amount"]).__name__)

30 records
{'order_id': 'KR4200', 'customer_id': 'C1645', 'segment': 'Retail-Core', 'amount': '4500', 'status': 'returned', 'order_date': '2026-08-03', 'discount': ''}

every value is a str


In [10]:
kit.check("thirty rows came out of the file", len(orders) == 30)
kit.check("the keys came from the header row",
          set(orders[0]) == {"order_id", "customer_id", "segment", "amount", "status",
                             "order_date", "discount"})
kit.check("everything the CSV gave back is text, including the amounts",
          all(isinstance(v, str) for v in orders[0].values()))

`csv.DictReader` takes its keys from the first row of the file. Change the header spelling in the file and every `record["amount"]` in your code raises `KeyError`, which is why the header row is part of the contract and not decoration.

Monday you had one amount stored as text and it broke a comparison. Write those same records to CSV and that bug disappears, because now every amount is text. The defect was never fixed. It was hidden by the format.

In [11]:
kit.flow(["the wrong path", "read a CSV by name", "convert or reject", "one pass, two files out", "a different agreement", "when JSON goes wrong"], lit=2)

## Section 3: convert on purpose, reject with a reason

Section 2 plus one new element: your notebook 1 functions, unchanged, pointed at file data.

These are the same three functions you carved in notebook 1, copied across unchanged. Read them and confirm that nothing in any of them knows it is reading a file.

In [12]:
def normalise_amount(raw):
    """Convert an amount, or raise ValueError with the interpreter's own wording."""
    return int(raw)


def clean_record(record):
    """Return one record with its amount as a number, or raise ValueError saying what arrived."""
    keeper = dict(record)
    keeper["amount"] = normalise_amount(record["amount"])
    return keeper


def clean_records(rows):
    """Call clean_record on every row and keep the failures, each with its reason."""
    clean = []
    rejects = []
    for r in rows:
        try:
            clean.append(clean_record(r))
        except ValueError as e:
            rejects.append({"order_id": r["order_id"], "reason": str(e)})
    return clean, rejects

clean, rejects = clean_records(orders)
print(f"input {len(orders)}, clean {len(clean)}, rejected {len(rejects)}")
for row in rejects:
    print(row)

input 30, clean 28, rejected 2
{'order_id': 'KR4210', 'reason': "invalid literal for int() with base 10: 'twelve'"}
{'order_id': 'KR4214', 'reason': "invalid literal for int() with base 10: ''"}


### Milestone: where this shows up in production

Public Health England, October 2020, dropped 15,841 COVID cases from reporting. A CSV was converted into an old Excel format that has a hard row limit, and the rows past the limit were silently discarded. Contact tracing never saw those people.

Nobody wrote bad code that day. Somebody did not know the format's contract.

### Interview question this milestone just made answerable

"Everything read from a CSV is a string. What breaks, and where do you convert?"

Answer in three beats: comparisons and arithmetic break first, conversion belongs in one named function, and every failed conversion becomes a rejection with a reason.

In [13]:
kit.flow(["the wrong path", "read a CSV by name", "convert or reject", "one pass, two files out", "a different agreement", "when JSON goes wrong"], lit=3)

## Section 4: one pass, two files out

Section 3 plus one new element: the results leave memory.

Shipping only the clean file is shipping half the job. The rejects file is what lets somebody else fix the source.

In [14]:
FIELDS = ["order_id", "customer_id", "segment", "amount", "status", "order_date", "discount"]

clean_path = f"{OUTPUT_DIR}/C2_W01_D02_clean_STUDENT.csv"
rejects_path = f"{OUTPUT_DIR}/C2_W01_D02_rejects_STUDENT.csv"

with open(clean_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=FIELDS)
    writer.writeheader()
    writer.writerows(clean)

with open(rejects_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["order_id", "reason"])
    writer.writeheader()
    writer.writerows(rejects)

print("wrote", clean_path)
print("wrote", rejects_path)

wrote output/C2_W01_D02_clean_STUDENT.csv
wrote output/C2_W01_D02_rejects_STUDENT.csv


`DictWriter` makes you state the field names on the way out. That is you writing the agreement for whoever reads this file next.

The `with` block guarantees the file closes when the block ends, including when your code raises inside it. On your laptop you get away with forgetting. On a server running this a thousand times a day you run out of file handles.

### Writing is not finishing. Reopening is finishing.

In [15]:
with open(clean_path) as f:
    reopened_clean = list(csv.DictReader(f))
with open(rejects_path) as f:
    reopened_rejects = list(csv.DictReader(f))

print(f"{len(orders)} in = {len(reopened_clean)} clean + {len(reopened_rejects)} rejected")
assert len(reopened_clean) + len(reopened_rejects) == len(orders), "records went missing"
print("reconciled")
print("first rejection as it reads from disk:", reopened_rejects[0])

30 in = 28 clean + 2 rejected
reconciled
first rejection as it reads from disk: {'order_id': 'KR4210', 'reason': "invalid literal for int() with base 10: 'twelve'"}


In [16]:
kit.check("both files reopened and parsed", len(reopened_clean) + len(reopened_rejects) > 0)
kit.check("the two files together account for every input row",
          len(reopened_clean) + len(reopened_rejects) == len(orders),
          f"{len(reopened_clean)} clean plus {len(reopened_rejects)} rejected")
kit.check("the amounts came back as text, because that is all a CSV agrees to",
          all(isinstance(r["amount"], str) for r in reopened_clean))

### What a CSV agrees to, and what JSON agrees to

In [17]:
kit.matrix(["CSV", "JSON"],
           ["Types it keeps", "Nesting it keeps", "What breaks it"],
           [["none, everything is text", "none, one row is flat",
             "a renamed header, silently"],
            ["numbers, text, true, false, null", "objects inside objects",
             "one missing bracket, wholly"]],
           title="two agreements about structure")

In [18]:
kit.flow(["the wrong path", "read a CSV by name", "convert or reject", "one pass, two files out", "a different agreement", "when JSON goes wrong"], lit=4)

## Section 5: the same records, a different agreement

Section 4 plus one new element: a format that agrees about types and nesting.

JSON agrees about more than CSV does. Numbers stay numbers, `null` stays null, and a value can hold another whole record.

In [19]:
with open(ORDERS_JSON) as f:
    json_orders = json.load(f)

print("amount type from JSON:", type(json_orders[0]["amount"]).__name__)
print("amount type from CSV: ", type(orders[0]["amount"]).__name__)
print()
print(json.dumps(json_orders[14], indent=2))

amount type from JSON: int
amount type from CSV:  str

{
  "order_id": "KR4214",
  "segment": "Student",
  "status": "delivered",
  "order_date": "2026-08-10",
  "amount": null,
  "source": {
    "system": "kalpa_retail_orders",
    "amount_raw": "2840"
  },
  "customer": {
    "customer_id": "C1517",
    "city": "Bengaluru",
    "signup_date": "2026-08-10"
  }
}


In [20]:
kit.check("JSON gave back real numbers rather than text",
          isinstance(json_orders[1]["amount"], int))
kit.check("the two unusable amounts arrive as null rather than as a spelled word",
          len([r for r in json_orders if r["amount"] is None]) == 2)

Look at order KR4214. Its amount was an empty cell in the CSV, so your cleaning run rejected it. The JSON still carries the original value one level down, inside `source`.

The record was never unrecoverable. The CSV export threw the value away.

### The flattening cost

To put that record into a CSV you have to choose: drop `source`, or invent a column such as `source_amount_raw`. Either way the shape changes, and the person downstream has to be told. That conversation is the cost of flattening.

In [21]:
missing_amount = [r for r in json_orders if r["amount"] is None]

for r in missing_amount:
    raw = r["source"]["amount_raw"]
    try:
        print(f'id {r["order_id"]}: recovered {normalise_amount(raw)} from the nested block')
    except ValueError:
        print(f'id {r["order_id"]}: nested block holds {raw!r}, which still will not convert')

id KR4210: nested block holds 'twelve', which still will not convert
id KR4214: recovered 2840 from the nested block


### Writing JSON back out

`json.dump` is the mirror of `json.load`. It is worth seeing once, because it is the only way to hand on a record that has a nested block inside it.

In [22]:
rejects_json_path = f"{OUTPUT_DIR}/C2_W01_D02_rejects_STUDENT.json"

with open(rejects_json_path, "w") as f:
    json.dump(rejects, f, indent=2)

with open(rejects_json_path) as f:
    reopened = json.load(f)

print("wrote and reopened", rejects_json_path)
print(reopened)

wrote and reopened output/C2_W01_D02_rejects_STUDENT.json
[{'order_id': 'KR4210', 'reason': "invalid literal for int() with base 10: 'twelve'"}, {'order_id': 'KR4214', 'reason': "invalid literal for int() with base 10: ''"}]


`indent=2` is for the human who opens the file next. Without it the whole file is one line, which parses perfectly well and reads terribly.

Notice what you did not have to do: no field names on the way out, because JSON carries the shape with it. That is the same agreement working in your favour for once.

In [23]:
kit.flow(["the wrong path", "read a CSV by name", "convert or reject", "one pass, two files out", "a different agreement", "when JSON goes wrong"], lit=5)

## Section 6: when JSON goes wrong

A JSON file is either wholly valid or wholly unreadable. There is no half-parsed JSON, which is the price of the stronger agreement.

The file below is a vendor feed whose transfer was cut off. Run it and read where the parser says it stopped.

In [24]:
def load_the_truncated_feed():
    with open(TRUNCATED_JSON) as f:
        return json.load(f)

show_failure(load_the_truncated_feed)

Traceback (most recent call last):
  File "/tmp/ipykernel_5185/1347761746.py", line 20, in show_failure
    fn()
  File "/tmp/ipykernel_5185/28563854.py", line 3, in load_the_truncated_feed
    return json.load(f)
           ^^^^^^^^^^^^
  File "/usr/lib/python3.11/json/__init__.py", line 293, in load
    return loads(fp.read(),
           ^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.11/json/__init__.py", line 346, in loads
    return _default_decoder.decode(s)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.11/json/decoder.py", line 337, in decode
    obj, end = self.raw_decode(s, idx=_w(s, 0).end())
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.11/json/decoder.py", line 353, in raw_decode
    obj, end = self.scan_once(s, idx)
               ^^^^^^^^^^^^^^^^^^^^^^
json.decoder.JSONDecodeError: Expecting ',' delimiter: line 48 column 1 (char 1027)



The parser names a line and a column. Go there first, every time.

Your mid-session exercise is this file. Open it, go to the line the error names, and write down what you find. The answer is more interesting than a bad character.

In [25]:
with open(TRUNCATED_JSON) as f:
    lines = f.read().splitlines()
print("the file has", len(lines), "lines")
print("last two lines:")
for n, line in enumerate(lines[-2:], start=len(lines) - 1):
    print(f"  {n}: {line!r}")

the file has 47 lines
last two lines:
  46: '      "city": "Singapore",'
  47: '      "signup_date": "2026-08-10"'


In [26]:
kit.check("the vendor feed is 47 lines and stops mid-record", len(lines) == 47,
          f"{len(lines)} lines")
kit.check("json.load refuses the whole file rather than half of it",
          True, "there is no half-parsed JSON")

### Where the parser was when it gave up

In [27]:
kit.vflow(["the file opens and parses fine for 47 lines",
           "line 48 never arrives",
           "the parser is still inside an object",
           "it reports line 48 column 1, which is past the end",
           "so you open the file at the last line it did read"],
          lit=3, title="reading a JSONDecodeError")

### Interview question this milestone just made answerable

"The vendor's JSON fails at line 47 column 5. What is your first move?"

Open the file at that line. Then decide whether the defect is in the file or in your assumption about the file. Repairing someone else's feed by hand is the last resort, never the first.

### What this notebook established

In [28]:
kit.table(
    ["The idea", "What proved it here"],
    [["A relative path is counted from the notebook's own folder", "os.getcwd() ended in notebooks"],
     ["Everything a CSV agrees to is text", "every value from DictReader was a str"],
     ["JSON keeps types and nesting, and breaks wholly", "the 47-line feed gave a line and a column"],
     ["A pass ships two files", "clean plus rejects reopened and reconciled against the input"]],
    caption="Day 2, notebook 2",
)
kit.flow(["the wrong path", "read a CSV by name", "convert or reject", "one pass, two files out", "a different agreement", "when JSON goes wrong"], lit=5, title="the notebook, end to end")
kit.check_summary()

The idea,What proved it here
A relative path is counted from the notebook's own folder,os.getcwd() ended in notebooks
Everything a CSV agrees to is text,every value from DictReader was a str
"JSON keeps types and nesting, and breaks wholly",the 47-line feed gave a line and a column
A pass ships two files,clean plus rejects reopened and reconciled against the input


## Crux

A file format is an agreement about structure, and everything a CSV agrees to is text. Your job at the boundary is to convert on purpose, reject with a reason, and hand on two files instead of one.

## What tomorrow does with this

Tomorrow hands you the full dataset at its dirtiest, and you point `clean_record` and `clean_records` at it without one edit. The question becomes how many usable records that dataset actually has.

Keep the three functions, not the output files. Tomorrow supplies its own data and calls your code.